# 01 · Load your image data

This notebook turns a zip file of images into a clean, fixed split
that the rest of your project uses. Run it **once** at the start of
the project, and again only if your data changes.

**Before you start**

- Your images are in one zip file, `data.zip`, in a folder of your
  Google Drive called `csc451_project` (README, Step 2).
- You know where your labels are: in folder names, in file names, or
  in a CSV file (README, Step 2).

**What it does**

| Step | What happens |
|---|---|
| 1–3 | Setup, settings, unzip |
| 4–5 | Find every image and its label |
| 6–7 | Remove files that will not open, and exact duplicates |
| 8 | Count the images in each class |
| 9–10 | Split into train / validation / test and **save the split** |
| 11–12 | Look at the images and at one batch |

**What it saves** in `csc451_project/outputs/` on your Drive:

- `splits.csv`: one row per image (path, label, split). Notebook 02
  reads it, so every run trains and tests on the same images.
- `classes.json`: the class names in index order. The app needs it.
- `data_info.json`: where the images are inside the zip.

Run the cells in order, from top to bottom. Change only the cells
marked **CHANGE**.

## Step 1 · Setup
Run this cell first, every time you open the notebook. In Colab it connects your Google Drive (click **Connect** and allow access) and downloads your team's repo so `src/` can be imported.

In [ ]:
# Run this cell first, every time you open the notebook.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# CHANGE: the address of YOUR team's copy of this repo on GitHub
REPO_URL = "https://github.com/dsc-scsu/csc451-project-starter.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git pull -q
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # on your own computer: work from the repo folder

sys.path.insert(0, "src")
print("Working in", Path.cwd())

## Step 2 · Settings (CHANGE)

| Setting | What to put |
|---|---|
| `PROJECT_FOLDER` | The folder in your Google Drive that holds `data.zip` |
| `DATA_ZIP_NAME` | The name of your zip file |
| `LABEL_SOURCE` | `"folders"`: `cat/001.jpg` · `"filenames"`: `cat_001.jpg` · `"csv"`: a CSV lists file and label |
| `FILENAME_PATTERN` | Only for `"filenames"`. The part in `( )` is the label. The default keeps everything before the last `_` and number |
| `CSV_NAME`, `CSV_FILE_COLUMN`, `CSV_LABEL_COLUMN` | Only for `"csv"`: the CSV's name inside the zip, and its column names |
| `CSV_IMAGE_FOLDER` | Only for `"csv"`: the folder, inside the zip, that the file names in the CSV are relative to (`""` = next to the CSV) |
| `GROUP_SOURCE` | `None`, `"filename"` or `"csv"`. Set it when one subject (a person, a pet, a plant) appears in several photos. See Step 9 |
| `VAL_SIZE`, `TEST_SIZE` | The share of images for validation and test. 0.15 each is a good start |
| `OVERWRITE_SPLITS` | Leave `False`. The split must not change once you start training |

In [ ]:
# CHANGE: settings for your project
PROJECT_FOLDER = "csc451_project"   # folder in your Google Drive
DATA_ZIP_NAME = "data.zip"          # your zipped images

LABEL_SOURCE = "folders"            # "folders", "filenames" or "csv"

# Only for LABEL_SOURCE = "filenames"
FILENAME_PATTERN = r"(.+)_\d+$"     # basset_hound_71.jpg -> basset_hound

# Only for LABEL_SOURCE = "csv"
CSV_NAME = "labels.csv"
CSV_FILE_COLUMN = "filename"
CSV_LABEL_COLUMN = "label"
CSV_IMAGE_FOLDER = ""               # "" = images are next to the CSV

# Optional: keep each subject's photos in one split (Step 9)
GROUP_SOURCE = None                 # None, "filename" or "csv"
CSV_GROUP_COLUMN = "subject"        # only for GROUP_SOURCE = "csv"

VAL_SIZE = 0.15
TEST_SIZE = 0.15
SEED = 0
OVERWRITE_SPLITS = False

In [ ]:
# Where things are. You should not need to change this cell.
if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    DATA_DIR = Path("/content/data")      # fast disk, erased after
else:                                     # the session ends
    PROJECT = Path("project_files")       # on your own computer
    DATA_DIR = PROJECT / "unzipped"
OUT = PROJECT / "outputs"                 # results to keep, on Drive
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)

import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader

import data_utils as du
import train_utils as tu

## Step 3 · Unzip the data
The zip is copied out of Drive and unzipped onto Colab's own disk, which is much faster to read during training. Colab erases this disk when the session ends; this cell unzips again next time.

The folder listing lets you check that the zip holds what you expect.

In [ ]:
du.unzip_once(PROJECT / DATA_ZIP_NAME, DATA_DIR)
IMAGE_ROOT = du.find_image_root(DATA_DIR)
print("Images are under:", IMAGE_ROOT)
print()
du.show_folders(IMAGE_ROOT)

## Step 4 · Find every image
This lists every `.jpg`, `.jpeg`, `.png`, `.bmp` and `.webp` file, in any subfolder, and skips hidden files such as `.DS_Store`.

In [ ]:
paths = du.list_images(IMAGE_ROOT)
print(len(paths), "image files found")
for p in paths[:5]:
    print("  ", p.relative_to(IMAGE_ROOT))

if len(paths) == 0:
    print("No images found. Look at the folder listing in Step 3.")

## Step 5 · Get one label per image
The label comes from the place you chose in `LABEL_SOURCE`. Images with no label (a file name that does not match the pattern) are listed and dropped.

In [ ]:
if LABEL_SOURCE == "folders":
    labels = du.labels_from_folders(paths)

elif LABEL_SOURCE == "filenames":
    labels = du.labels_from_filenames(paths, FILENAME_PATTERN)

elif LABEL_SOURCE == "csv":
    found = sorted(IMAGE_ROOT.rglob(CSV_NAME))
    if not found:
        raise FileNotFoundError(f"{CSV_NAME} is not in the zip")
    csv_file = found[0]
    csv_root = csv_file.parent / CSV_IMAGE_FOLDER
    paths, labels = du.labels_from_csv(
        csv_file, csv_root, CSV_FILE_COLUMN, CSV_LABEL_COLUMN)

else:
    raise ValueError('LABEL_SOURCE must be "folders", "filenames" or "csv"')

no_label = [p for p, y in zip(paths, labels) if y is None]
if no_label:
    print(len(no_label), "files have no label and are dropped, e.g.:")
    for p in no_label[:5]:
        print("  ", p.name)
keep = [i for i, y in enumerate(labels) if y is not None]
paths = [paths[i] for i in keep]
labels = [labels[i] for i in keep]

print(len(paths), "images with a label")
print("Classes:", sorted(set(labels)))

## Step 6 · Remove files that will not open
One broken file stops training in the middle of an epoch, so check every file once, now. Grayscale images and PNGs with transparency are fine: the Dataset converts every image to 3 channels (RGB).

In [ ]:
bad = set(du.find_bad_images(paths))
print(len(bad), "files will not open")
for p in list(bad)[:5]:
    print("  ", p.relative_to(IMAGE_ROOT))

keep = [i for i, p in enumerate(paths) if p not in bad]
paths = [paths[i] for i in keep]
labels = [labels[i] for i in keep]

## Step 7 · Remove exact duplicates
The same file saved twice (often under two names) is a common leak: if one copy lands in training and the other in the test set, the model is tested on an image it has already seen, and the test score is too high.

In [ ]:
copies = set(du.find_duplicates(paths))
print(len(copies), "files are exact copies of another file")
for p in list(copies)[:5]:
    print("  ", p.relative_to(IMAGE_ROOT))

keep = [i for i, p in enumerate(paths) if p not in copies]
paths = [paths[i] for i in keep]
labels = [labels[i] for i in keep]
print(len(paths), "images left")

## Step 8 · Count the images in each class
If one class is much larger than the others, a model that always answers that class already gets a high accuracy. That is your **baseline**, and your model has to beat it. Notebook 02 computes it.

In [ ]:
counts = pd.Series(labels).value_counts()
print(counts.to_string())
print()
print(f"Largest class: {counts.max() / counts.sum():.1%} of the images")

counts.sort_values().plot.barh(figsize=(6, 0.4 * len(counts) + 1))
plt.xlabel("images")
plt.title("Images per class")
plt.show()

if counts.min() < 20:
    print("Warning: some classes have fewer than 20 images.")
    print("Collect more, or merge or drop those classes.")

## Step 9 · Groups (only if one subject is in several photos)

If the same person, pet, plant or object appears in several photos,
a random split puts some of its photos in training and some in the
test set. The model then recognizes the subject, not the class, and
the test score is too high.

To prevent this, give every photo a **group** (who or what is in it).
All photos of one group then go to the same split.

- `GROUP_SOURCE = "filename"`: the group is the part of the file name
  before the first `_`, so `rex_03.jpg` belongs to group `rex`.
  Rename your files this way, or edit `du.group_from_filename`.
- `GROUP_SOURCE = "csv"`: the group is in the CSV column
  `CSV_GROUP_COLUMN`.
- `GROUP_SOURCE = None`: every photo is a different subject.

In [ ]:
groups = None
if GROUP_SOURCE == "filename":
    groups = du.group_from_filename(paths)
elif GROUP_SOURCE == "csv":
    table_csv = pd.read_csv(csv_file)
    name_to_group = dict(zip(table_csv[CSV_FILE_COLUMN].astype(str),
                             table_csv[CSV_GROUP_COLUMN].astype(str)))
    groups = [name_to_group[str(p.relative_to(csv_root))] for p in paths]

if groups is not None:
    print(len(set(groups)), "groups for", len(paths), "images")

table = du.make_table(paths, labels, IMAGE_ROOT, groups)
table.head()

## Step 10 · Split into train / validation / test, and save

- **train**: the model learns from these images.
- **val** (validation): you use these to compare settings and to
  choose the best epoch.
- **test**: used **once**, at the very end, for the score you report.

The split is saved to `outputs/splits.csv`. It is not made again
unless you set `OVERWRITE_SPLITS = True`, because changing the split
in the middle of a project makes your results impossible to compare.

In [ ]:
splits_file = OUT / "splits.csv"

if splits_file.exists() and not OVERWRITE_SPLITS:
    print("Using the split saved earlier:", splits_file)
    table = pd.read_csv(splits_file)
else:
    table = du.make_splits(table, VAL_SIZE, TEST_SIZE,
                           by_group=groups is not None, seed=SEED)
    table.to_csv(splits_file, index=False)
    print("Saved", splits_file)

classes = sorted(table["label"].unique())
with open(OUT / "classes.json", "w") as f:
    json.dump(classes, f)
with open(OUT / "data_info.json", "w") as f:
    json.dump({"image_root": str(IMAGE_ROOT.relative_to(DATA_DIR))}, f)

print()
print(pd.crosstab(table["label"], table["split"], margins=True))

if "group" in table.columns:
    sets = [set(table.loc[table["split"] == s, "group"])
            for s in ["train", "val", "test"]]
    shared = (sets[0] & sets[1]) | (sets[0] & sets[2]) | (sets[1] & sets[2])
    print()
    print("Groups in more than one split:", len(shared), "(should be 0)")

missing = pd.crosstab(table["label"], table["split"]) == 0
if missing.to_numpy().any():
    print()
    print("Warning: some classes have no images in some split.")
    print("Collect more images (or more groups) for those classes.")

## Step 11 · Look at the images
A few training images from each class. Check that the labels are right and that nothing odd (a watermark, a border, a different background in one class) could let the model cheat.

In [ ]:
train_table = table[table["split"] == "train"]
n_show = 5
fig, axes = plt.subplots(len(classes), n_show,
                         figsize=(2 * n_show, 2 * len(classes)),
                         squeeze=False)
for row, c in enumerate(classes):
    rows = train_table[train_table["label"] == c].head(n_show)
    for col in range(n_show):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(rows):
            img = Image.open(IMAGE_ROOT / rows["path"].iloc[col])
            ax.imshow(img.convert("RGB"))
        if col == 0:
            ax.set_title(c, loc="left")
plt.tight_layout()
plt.show()

## Step 12 · Check one batch
This builds the training Dataset and DataLoader exactly as notebook 02 does, and checks the shapes, the value range, and the labels in one batch. The grid shows the images after augmentation: if a transform ruins the subject, you see it here.

In [ ]:
train_tfms, eval_tfms = tu.make_transforms(image_size=224, flip=True)
train_ds = du.ImageDataset(train_table, IMAGE_ROOT, classes, train_tfms)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)

x, y = next(iter(train_dl))
print("images:", tuple(x.shape), " labels:", tuple(y.shape))
print(f"pixel values from {x.min():.2f} to {x.max():.2f}")
print("labels in this batch:", y.tolist())

mean = torch.tensor(tu.IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(tu.IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, img, label in zip(axes.flat, x, y):
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(classes[label], fontsize=9)
    ax.axis("off")
plt.show()

## Done

Your split is saved. Next: **02_train_image_model.ipynb**.

Your data is a table instead of images? Use **03_table_data.ipynb**.